In [ ]:
# 모델 이름

from lib.utils.model_list import display_model_list

display_model_list()

In [ ]:
import pandas as pd
import pytorch_lightning as pl
from torch.utils.data.dataloader import DataLoader
from torchvision.transforms import Compose, Normalize, Resize, ToTensor

from dataset.slope_dataset import SlopeDataset
from lib.utils.path import output_path
from models.wheel_safe_classifier import WheelSafeClassifier

# 이미지 크기는 모델에 맞게 조절 (EfficientNet 기본 224, 240 등)
data_transform = Compose(
    [
        Resize((224, 224)),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# CSV 또는 DataFrame 로드 (올려주신 이미지 기준 예시)
df = pd.read_csv(output_path() / 'slope_labels_007.csv')

train_dataset = SlopeDataset(df, transform=data_transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# --- 모델 생성 및 학습 ---
model_system = WheelSafeClassifier(model_name='efficientnet_b0')

trainer = pl.Trainer(
    max_epochs=50,
    accelerator='gpu',
    devices=1,
    precision=16,  # 경사도 연산 가속
)

trainer.fit(model_system, train_loader)

In [ ]:
import matplotlib.pyplot as plt

from lib.utils.path import data_path
from run.predict import execute


def show_image(image, angle):
    img = plt.imread(image)
    plt.imshow(img)
    plt.title(f'Predicted Angle: {angle:.2f}°')
    plt.axis('off')
    plt.show()


execute('efficientnet_b0', data_path() / 'test', show_image)

In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd

from lib.utils.path import output_path


def change_root(src, dest):
    files = glob.glob(os.path.join(output_path(), 'slope_labels_total_test_30%.csv'))

    src_path = Path(src)
    dest_path = Path(dest)

    for file_path in files:
        df = pd.read_csv(file_path, index_col=0)
        print(f'변경 전 첫 행: {df["path"].iloc[0]}')
        df['path'] = (
            df['path']
            .apply(lambda x: str(Path(x)))
            .str.replace(str(src_path), str(dest_path), regex=False)
        )
        df['path'] = df['path'].str.replace(src, dest, regex=False)
        print(f'변경 후 첫 행: {df["path"].iloc[0]}')
        # df['path'] = df['path'].str.replace(src, dest, regex=False)
        p = Path(file_path)
        df.to_csv(f'{output_path()}/{p.stem}_replaced{p.suffix}')


pre_path = [
    'c:\wheel-safe',
    'c:\Workspaces\projects\wheel-safe',
    'c:\potenup3\deeplearning\wheel-safe',
]
# for path in pre_path:
change_root(
    'c:\Workspaces\projects\wheel-safe', '/Users/kyungpyokim/Projects/wheel-safe'
)


In [ ]:
import re

import pandas as pd


def change_root_all(pre_paths, dest):
    # 파일 찾기
    files = glob.glob(os.path.join(output_path(), 'slope_labels_total_train_70%.csv'))

    # 1. 모든 후보 경로를 정규식의 OR(|) 패턴으로 묶음
    # re.escape는 경로에 포함된 '.' 이나 '\'를 문자로 인식하게 해줍니다.
    pattern = '|'.join([re.escape(p) for p in pre_paths])

    # 2. 데이터프레임 내의 경로 구분자 표준화 (역슬래시를 슬래시로)
    # 윈도우/맥 혼용 시 발생할 수 있는 문제를 방지합니다.
    dest_fixed = dest.replace('\\', '/')

    for file_path in files:
        df = pd.read_csv(file_path, index_col=0)

        # 원본 데이터의 역슬래시를 모두 슬래시로 미리 통일 (선택 사항이지만 안전함)
        df['path'] = df['path'].str.replace('\\', '/', regex=False)

        # 한방에 치환 (정규식 사용)
        # pre_paths 내부의 경로들도 슬래시 기준으로 정리해서 비교
        clean_pattern = '|'.join([re.escape(p.replace('\\', '/')) for p in pre_paths])

        df['path'] = df['path'].str.replace(clean_pattern, dest_fixed, regex=True)

        # 저장
        p = Path(file_path)
        output_name = f'{output_path()}/{p.stem}_replaced{p.suffix}'
        df.to_csv(output_name)
        print(f'처리 완료: {output_name}')


# 사용 예시 (반드시 r을 붙여서 Raw string으로 선언하세요!)
pre_paths = [
    r'c:\wheel-safe',
    r'C:\wheel-safe',
    r'c:\Workspaces\projects\wheel-safe',
    r'C:\Workspaces\projects\wheel-safe',
    r'c:\potenup3\deeplearning\wheel-safe',
    r'C:\potenup3\deeplearning\wheel-safe',
]

change_root_all(pre_paths, '/Users/kyungpyokim/Projects/wheel-safe')

In [ ]:
import pandas as pd


def change_root_all_robust(pre_paths, dest):
    # 파일 경로 설정
    files = glob.glob(
        os.path.join(output_path(), 'slope_labels_total_up_to_train_70%.csv')
    )

    # 1. 모든 후보 경로를 정규식 패턴으로 묶기 (슬래시 방향 통일)
    # 각 경로를 정규식 특수문자 처리(escape)하고 슬래시로 통일합니다.
    clean_paths = [re.escape(p.replace('\\', '/')) for p in pre_paths]
    pattern = '|'.join(clean_paths)

    # 2. 목적지 경로도 슬래시로 통일
    dest_fixed = dest.replace('\\', '/')

    for file_path in files:
        df = pd.read_csv(file_path, index_col=0)

        # 데이터프레임 내의 모든 역슬래시(\)를 슬래시(/)로 먼저 변경
        df['path'] = df['path'].str.replace('\\', '/', regex=False)

        # 대소문자 구분 없이(flags=re.IGNORECASE) 패턴에 맞는 부분을 dest로 치환
        df['path'] = df['path'].str.replace(
            pattern,
            dest_fixed,
            regex=True,
            flags=re.IGNORECASE,  # <-- 핵심: C:\와 c:\를 모두 잡습니다.
        )

        # 결과 저장
        p = Path(file_path)
        output_name = f'{output_path()}/{p.stem}_replaced{p.suffix}'
        df.to_csv(output_name)
        print(f'처리 완료: {output_name}')


# 호출부 (반드시 r을 붙여주세요)
pre_paths = [
    r'c:\wheel-safe',
    r'c:\Workspaces\projects\wheel-safe',
    r'c:\potenup3\deeplearning\wheel-safe',
]

change_root_all_robust(pre_paths, '/Users/kyungpyokim/Projects/wheel-safe')

In [ ]:
df = pd.read_csv(output_path() / 'slope_labels_total_test_30%.csv')
len(df)
df = pd.read_csv(output_path() / 'slope_labels_total_up_to_train_70%.csv')
len(df)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 데이터 불러오기
file_path = output_path() / 'slope_labels_total_test_30%_replaced.csv'
df = pd.read_csv(file_path)

# 2. 5:5 (50%) 비율로 분할 (shuffle=True를 통해 데이터를 골고루 섞어줍니다)
df1, df2 = train_test_split(df, test_size=0.5, random_state=42, shuffle=True)

# 3. 각각 새로운 파일로 저장
df1.to_csv('slope_labels_50_part1.csv', index=False)
df2.to_csv('slope_labels_50_part2.csv', index=False)

print(f'전체 행 수: {len(df)}')
print(f'Part 1 행 수: {len(df1)}')
print(f'Part 2 행 수: {len(df2)}')